# FD004 multi-prefix EOL smoothing v01

This notebook evaluates multi-prefix / EOL smoothing using existing internal-validation artificial-cutoff predictions only. It does not use final test data and does not train a new large model.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "results" / "FD004" / "fd004_multiprefix_eol_smoothing_results_v01.csv").exists():
        PROJECT_ROOT = candidate
        break

results = pd.read_csv(PROJECT_ROOT / "results/FD004/fd004_multiprefix_eol_smoothing_results_v01.csv")
predictions = pd.read_csv(PROJECT_ROOT / "results/FD004/fd004_multiprefix_predictions_v01.csv")
best = pd.read_csv(PROJECT_ROOT / "results/FD004/fd004_multiprefix_best_candidate_v01.csv")
bins = pd.read_csv(PROJECT_ROOT / "results/FD004/fd004_multiprefix_metrics_by_rul_bin_v01.csv")
diagnostics = pd.read_csv(PROJECT_ROOT / "results/FD004/fd004_multiprefix_diagnostics_v01.csv")


## Candidate comparison and diagnostics


In [ ]:
display(results.sort_values("CMAPSS_mean"))
display(diagnostics)

best_name = best.loc[0, "candidate_name"]
baseline_pred = predictions[predictions["candidate_name"].eq("baseline")].copy()
best_pred = predictions[predictions["candidate_name"].eq(best_name)].copy()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(baseline_pred["true_RUL"], baseline_pred["pred_RUL"], s=14, alpha=0.5, label="baseline")
ax.scatter(best_pred["true_RUL"], best_pred["pred_RUL"], s=14, alpha=0.5, label=best_name)
lim = max(baseline_pred["true_RUL"].max(), baseline_pred["pred_RUL"].max(), best_pred["pred_RUL"].max())
ax.plot([0, lim], [0, lim], color="black", linewidth=1)
ax.set_xlabel("True RUL")
ax.set_ylabel("Predicted RUL")
ax.set_title("True RUL vs predicted RUL")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(baseline_pred["true_RUL"], baseline_pred["error"], s=14, alpha=0.45, label="baseline")
ax.scatter(best_pred["true_RUL"], best_pred["error"], s=14, alpha=0.45, label=best_name)
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("True RUL")
ax.set_ylabel("Pred - true")
ax.set_title("Residual by true RUL")
ax.legend()
plt.show()

compare_bins = bins[bins["candidate_name"].isin(["baseline", best_name])].copy()
pivot_mae = compare_bins.pivot(index="rul_bin", columns="candidate_name", values="MAE")
display(pivot_mae)
pivot_mae.plot(kind="bar", figsize=(7, 4), title="MAE by RUL bin")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()

pivot_share = compare_bins.pivot(index="rul_bin", columns="candidate_name", values="CMAPSS_share")
display(pivot_share)
pivot_share.plot(kind="bar", figsize=(7, 4), title="CMAPSS share by RUL bin")
plt.ylabel("Share")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
baseline_pred["eol_baseline"].hist(ax=ax, bins=25, alpha=0.45, label="baseline EOL")
best_pred["eol_smoothed"].hist(ax=ax, bins=25, alpha=0.45, label=f"{best_name} smoothed EOL")
ax.set_title("Estimated failure-cycle distribution")
ax.set_xlabel("Estimated failure cycle")
ax.legend()
plt.show()

example_units = best_pred.sort_values("cmapss_penalty", ascending=False)[["split_id", "unit_id"]].drop_duplicates().head(4)
for _, row in example_units.iterrows():
    mask = best_pred["split_id"].eq(row["split_id"]) & best_pred["unit_id"].eq(row["unit_id"])
    bmask = baseline_pred["split_id"].eq(row["split_id"]) & baseline_pred["unit_id"].eq(row["unit_id"])
    unit_best = best_pred.loc[mask].sort_values("cycle")
    unit_base = baseline_pred.loc[bmask].sort_values("cycle")
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(unit_base["cycle"], unit_base["true_RUL"], marker="o", label="true RUL")
    ax1.plot(unit_base["cycle"], unit_base["pred_RUL"], marker="o", label="baseline pred")
    ax1.plot(unit_best["cycle"], unit_best["pred_RUL"], marker="o", label=f"{best_name} pred")
    ax1.set_xlabel("cycle")
    ax1.set_ylabel("RUL")
    ax2 = ax1.twinx()
    ax2.plot(unit_best["cycle"], unit_best["eol_smoothed"], linestyle="--", color="tab:red", label="smoothed EOL")
    ax2.set_ylabel("Estimated failure cycle")
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc="best")
    ax1.set_title(f"Unit {int(row['unit_id'])} split {int(row['split_id'])}")
    plt.tight_layout()
    plt.show()

top_baseline = baseline_pred.sort_values("cmapss_penalty", ascending=False).head(10)
top_best = best_pred.sort_values("cmapss_penalty", ascending=False).head(10)
display(Markdown("### Top 10 CMAPSS baseline"))
display(top_baseline[["unit_id", "split_id", "cycle", "true_RUL", "pred_RUL", "error", "cmapss_penalty", "n_prefixes_used"]])
display(Markdown(f"### Top 10 CMAPSS {best_name}"))
display(top_best[["unit_id", "split_id", "cycle", "true_RUL", "pred_RUL", "error", "cmapss_penalty", "n_prefixes_used"]])


## Final decision cell


In [ ]:
row = best.iloc[0]
diag = diagnostics.iloc[0]
baseline = results[results["candidate_name"].eq("baseline")].iloc[0]
display(Markdown(
    f"""### Conclusion v01

    - Mejor candidato encontrado: `{row['candidate_name']}`.
    - Comparacion contra baseline: CMAPSS_mean {baseline['CMAPSS_mean']:.4f} -> {row['CMAPSS_mean']:.4f}; dangerous_20 {baseline['dangerous_20_pct']:.2f}% -> {row['dangerous_20_pct']:.2f}%; RMSE {baseline['RMSE']:.4f} -> {row['RMSE']:.4f}.
    - Recomendacion: {'adoptar' if bool(row['accepted']) else 'no adoptar todavia'}.
    - Riesgos metodologicos: esta version usa prefijos ya disponibles en cortes artificiales internos, con spacing observado de 30 ciclos, no una grilla exacta t-20.
    - Confirmacion: no se uso test final.
    - Confirmacion: para cada ciclo t solo se usaron prefijos del mismo motor/split con ciclo <= t. Check no_future_prefixes_used={bool(diag['no_future_prefixes_used'])}.
    """
))


## Decision final v01

- Mejor candidato encontrado: `eol_mean`.
- Comparacion contra baseline actual: CMAPSS mean `4.9531` -> `3.2217`, RMSE `16.8913` -> `16.3453`, dangerous_20 `6.69%` -> `0.40%`.
- Recomendacion: no adoptarlo todavia.
- Motivo: Not accepted: best smoothing did not satisfy all adoption guards (CMAPSS_improves=True, danger_ok=True, rmse_ok=True, low_rul_ok=False, enough_units=True).
- Riesgos metodologicos: esta version usa prefijos ya disponibles en cortes artificiales internos; el spacing observado es `30.0` ciclos, no una grilla exacta `t-20`.
- Confirmacion explicita: no se uso test final.
- Confirmacion explicita: para cada ciclo `t` solo se usaron prefijos del mismo motor/split con `cycle <= t`; `no_future_prefixes_used=True`.
